# Data Acquisition, Preprocessing, Tokenization & Vectorization for Symbolic-Music Composer Classification

## 1. Environment Setup

In [ ]:
# ---------------------------------------------------------------------------
# 1.1 Package installation (Kaggle kernels: run once per session)
# ---------------------------------------------------------------------------
import sys, subprocess

def pip_install(packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages])

pip_install(["music21", "pretty_midi", "gensim"])

# ---------------------------------------------------------------------------
# 1.2 Imports
# ---------------------------------------------------------------------------
import os
import re
import glob
import time
import pickle
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import music21
import pretty_midi

import nltk
from nltk.corpus import stopwords as nltk_stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

# NLTK resources needed for the *literal* text-preprocessing demonstrations
# (Section 3.2 metadata tokenization, Section 5.1 stopword reference list)
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.download(resource, quiet=True)
    except Exception:
        pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("music21:", music21.__version__)
print("gensim available:", Word2Vec is not None)

## 2. Data Acquisition

In [ ]:
# ---------------------------------------------------------------------------
# 2.1 Kaggle API authentication
# ---------------------------------------------------------------------------
import os

# Kaggle Credentials
os.environ["KAGGLE_USERNAME"] = "KGAT_5b91ead9128243f7620aff08a037f979"
os.environ["KAGGLE_KEY"] = "KGAT_5b91ead9128243f7620aff08a037f979"

# Install the kaggle CLI/package if it isn't already available
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])

print("Kaggle credentials configured for user:", os.environ.get("KAGGLE_USERNAME"))

In [ ]:
# ---------------------------------------------------------------------------
# 2.2 Download and unzip the dataset via the Kaggle API
# ---------------------------------------------------------------------------
import subprocess

DATA_DIR = "data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "blanderbuss/midi-classic-music",
     "-p", DATA_DIR,
     "--unzip"],
    capture_output=True, text=True
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(
        "Kaggle download failed. Common causes: (1) invalid/expired API key, "
        "(2) you have not accepted the dataset's terms on the Kaggle website "
        "(visit the dataset page and click 'Download' once manually to accept), "
        "(3) missing internet access — on Kaggle Notebooks, enable "
        "Settings -> Internet -> On."
    )

DATA_ROOT = DATA_DIR
print("\nDownload complete. Sample of files in", DATA_ROOT, ":")
print(os.listdir(DATA_ROOT)[:20])

In [ ]:
# ---------------------------------------------------------------------------
# 2.3 Discover and label MIDI files
# ---------------------------------------------------------------------------
import glob
import pandas as pd

COMPOSERS = ["Bach", "Beethoven", "Chopin", "Mozart"]

def discover_midi_files(root_dir, composers):
    """Recursively discover .mid/.midi files and label each by composer,
    matching either a per-composer subfolder structure or a composer name
    embedded in the filename/path."""
    all_midi = glob.glob(os.path.join(root_dir, "**", "*.mid"), recursive=True)
    all_midi += glob.glob(os.path.join(root_dir, "**", "*.midi"), recursive=True)

    records = []
    for path in all_midi:
        lower_path = path.lower()
        for composer in composers:
            if composer.lower() in lower_path:
                records.append({
                    "filepath": path,
                    "composer": composer,
                    "filename": os.path.basename(path),
                })
                break
    return pd.DataFrame(records)

file_manifest = discover_midi_files(DATA_ROOT, COMPOSERS)
print(f"Discovered {len(file_manifest)} labeled MIDI files.")
file_manifest["composer"].value_counts()

In [ ]:
# ---------------------------------------------------------------------------
# 2.4 Sampling cap for a tractable first pass (raise/remove for final run)
# ---------------------------------------------------------------------------
MAX_FILES_PER_COMPOSER = 80

file_manifest = (
    file_manifest.groupby("composer", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), MAX_FILES_PER_COMPOSER), random_state=SEED))
    .reset_index(drop=True)
)

print(f"Working corpus size after cap: {len(file_manifest)}")
file_manifest.to_csv("file_manifest.csv", index=False)
file_manifest.head()

## 3. Preprocessing Stage 1 — Tokenization

In [ ]:
# ---------------------------------------------------------------------------
# 3.1 MIDI -> symbolic token sequence (music21)
# ---------------------------------------------------------------------------
def tokenize_midi(filepath):
    """Tokenize a MIDI file into an ordered list of note/chord string tokens.

    Returns None if the file cannot be parsed (corrupted / malformed MIDI) —
    these are logged separately rather than silently dropped, for an honest
    accounting of data quality.
    """
    try:
        score = music21.converter.parse(filepath)
    except Exception:
        return None

    try:
        parts = music21.instrument.partitionByInstrument(score)
        elements = parts.parts[0].recurse() if parts and parts.parts else score.flat.notes
    except Exception:
        elements = score.flat.notes

    tokens = []
    for el in elements:
        if isinstance(el, music21.note.Note):
            tokens.append(str(el.pitch))
        elif isinstance(el, music21.chord.Chord):
            tokens.append('.'.join(str(p) for p in el.normalOrder))

    return tokens if tokens else None


tokenized_corpus = []
failed_files = []

for _, row in file_manifest.iterrows():
    toks = tokenize_midi(row["filepath"])
    if toks is None:
        failed_files.append(row["filepath"])
    else:
        tokenized_corpus.append({
            "filepath": row["filepath"],
            "composer": row["composer"],
            "tokens": toks,
        })

print(f"Tokenized {len(tokenized_corpus)} files successfully.")
print(f"Failed to parse {len(failed_files)} files.")

tokens_df = pd.DataFrame(tokenized_corpus)
tokens_df["n_tokens"] = tokens_df["tokens"].apply(len)
tokens_df[["composer", "n_tokens"]].groupby("composer").describe()

In [ ]:
# ---------------------------------------------------------------------------
# 3.1.1 Illustrate tokenization on one example file
# ---------------------------------------------------------------------------
example = tokens_df.iloc[0]
print(f"File: {example['filepath']}  (composer: {example['composer']})")
print(f"First 25 tokens:\n{example['tokens'][:25]}")

In [ ]:
# ---------------------------------------------------------------------------
# 3.2 Literal NLP tokenization on genuine text metadata (track/instrument names)
# ---------------------------------------------------------------------------
def extract_metadata_text(filepath):
    """Pull track/instrument names out of a MIDI file as free text, for a
    literal (non-analogical) demonstration of NLP tokenization."""
    try:
        pm = pretty_midi.PrettyMIDI(filepath)
    except Exception:
        return ""
    names = [instr.name for instr in pm.instruments if instr.name]
    return " ".join(names)

sample_paths = tokens_df["filepath"].sample(min(30, len(tokens_df)), random_state=SEED)
metadata_texts = [extract_metadata_text(p) for p in sample_paths]
metadata_texts = [t for t in metadata_texts if t.strip()]

print(f"Recovered non-empty text metadata from {len(metadata_texts)} sample files.\n")
if metadata_texts:
    print("Example raw metadata string:", repr(metadata_texts[0]))
    print("\nnltk.word_tokenize output:", word_tokenize(metadata_texts[0]))
else:
    print("No textual track metadata present in this sample; "
          "this is common for MIDI files exported without instrument labels. "
          "The literal-NLP demonstration is illustrative and does not gate "
          "the rest of the pipeline, which relies on the symbolic tokens.")

## 4. Preprocessing Stage 2 — Lowercasing

In [ ]:
# ---------------------------------------------------------------------------
# 4.1 Literal case-folding
# ---------------------------------------------------------------------------
def lowercase_tokens(tokens):
    return [t.lower() for t in tokens]

tokens_df["tokens_lower"] = tokens_df["tokens"].apply(lowercase_tokens)

# ---------------------------------------------------------------------------
# 4.2 Quantify enharmonic-spelling collapse (the musically meaningful analogue)
# ---------------------------------------------------------------------------
def pitch_class_of(token):
    """Map a note or chord token to its underlying pitch-class set, to
    measure how much surface-level 'spelling' variation exists."""
    try:
        if '.' in token:
            return token  # chord tokens are already pitch-class integers
        p = music21.pitch.Pitch(token)
        return str(p.pitchClass)
    except Exception:
        return token

vocab_before = {t for seq in tokens_df["tokens"] for t in seq}
vocab_after_case = {t for seq in tokens_df["tokens_lower"] for t in seq}
vocab_after_enharmonic = {pitch_class_of(t) for seq in tokens_df["tokens"] for t in seq}

print(f"Vocabulary size, raw tokens:                {len(vocab_before)}")
print(f"Vocabulary size, after case-folding:        {len(vocab_after_case)}  "
      f"(reduction: {len(vocab_before) - len(vocab_after_case)})")
print(f"Vocabulary size, after enharmonic collapse:  {len(vocab_after_enharmonic)}  "
      f"(reduction: {len(vocab_before) - len(vocab_after_enharmonic)})")

## 5. Preprocessing Stage 3 — Stopword Removal

In [ ]:
# ---------------------------------------------------------------------------
# 5.1 Literal NLP stopwords (reference only)
# ---------------------------------------------------------------------------
english_stopwords = set(nltk_stopwords.words("english"))
print(f"nltk English stopword list size: {len(english_stopwords)}")
print("Sample:", sorted(list(english_stopwords))[:15])

In [ ]:
# ---------------------------------------------------------------------------
# 5.2 Data-driven document-frequency stopword removal for music tokens
# ---------------------------------------------------------------------------
N_DOCS = len(tokens_df)

doc_freq = Counter()
for seq in tokens_df["tokens_lower"]:
    doc_freq.update(set(seq))  # count each token once per document

doc_freq_df = (
    pd.DataFrame(doc_freq.items(), columns=["token", "doc_count"])
    .assign(doc_freq=lambda d: d["doc_count"] / N_DOCS)
    .sort_values("doc_freq", ascending=False)
)

DF_THRESHOLD = 0.90  # token appears in >90% of all pieces -> low discriminative power
musical_stopwords = set(doc_freq_df.loc[doc_freq_df["doc_freq"] > DF_THRESHOLD, "token"])

print(f"Tokens exceeding DF > {DF_THRESHOLD:.0%}: {len(musical_stopwords)}")
doc_freq_df.head(15)

In [ ]:
# ---------------------------------------------------------------------------
# 5.3 Apply musical-stopword removal
# ---------------------------------------------------------------------------
def remove_stopwords(tokens, stopword_set):
    return [t for t in tokens if t not in stopword_set]

tokens_df["tokens_nostop"] = tokens_df["tokens_lower"].apply(
    lambda seq: remove_stopwords(seq, musical_stopwords)
)

before_avg_len = tokens_df["tokens_lower"].apply(len).mean()
after_avg_len = tokens_df["tokens_nostop"].apply(len).mean()

print(f"Average sequence length before stopword removal: {before_avg_len:.1f}")
print(f"Average sequence length after stopword removal:  {after_avg_len:.1f}")
print(f"Average tokens removed per document:              {before_avg_len - after_avg_len:.1f}")

## 6. Preprocessing Stage 4 — Stemming vs. Lemmatization

In [ ]:
# ---------------------------------------------------------------------------
# 6.1 MusicStemmer — rule-based, aggressive octave-folding
# ---------------------------------------------------------------------------
def music_stem(token):
    """Rule-based reduction: strip the octave digit from a note token, or
    reduce a chord token to its sorted, deduplicated pitch-class set.
    Purely syntactic — no music-theoretic lookup, analogous to Porter
    stemming's mechanical suffix stripping."""
    if '.' in token:
        pcs = sorted(set(int(p) for p in token.split('.')))
        return '.'.join(str(p) for p in pcs)
    return re.sub(r'\d+$', '', token)  # e.g. "C#4" -> "C#"


# ---------------------------------------------------------------------------
# 6.2 MusicLemmatizer — knowledge-based, music21 prime-form canonicalization
# ---------------------------------------------------------------------------
def music_lemmatize(token):
    """Knowledge-based reduction using music-theoretic set-class identity.
    Chords are mapped to their prime form (transposition/inversion
    invariant canonical form); notes are mapped to pitch class. Falls back
    to the raw token if music21 cannot parse it."""
    try:
        if '.' in token:
            pcs = [int(p) for p in token.split('.')]
            c = music21.chord.Chord(pcs)
            return '.'.join(str(p) for p in c.primeForm)
        else:
            p = music21.pitch.Pitch(token)
            return str(p.pitchClass)
    except Exception:
        return token


tokens_df["tokens_stemmed"] = tokens_df["tokens_nostop"].apply(
    lambda seq: [music_stem(t) for t in seq]
)
tokens_df["tokens_lemmatized"] = tokens_df["tokens_nostop"].apply(
    lambda seq: [music_lemmatize(t) for t in seq]
)

vocab_nostop = {t for seq in tokens_df["tokens_nostop"] for t in seq}
vocab_stemmed = {t for seq in tokens_df["tokens_stemmed"] for t in seq}
vocab_lemmatized = {t for seq in tokens_df["tokens_lemmatized"] for t in seq}

print(f"Vocabulary size before stemming/lemmatization: {len(vocab_nostop)}")
print(f"Vocabulary size after stemming (aggressive):    {len(vocab_stemmed)}")
print(f"Vocabulary size after lemmatization (canonical): {len(vocab_lemmatized)}")

In [ ]:
# ---------------------------------------------------------------------------
# 6.3 Side-by-side example: raw -> stemmed -> lemmatized
# ---------------------------------------------------------------------------
sample_tokens = tokens_df.iloc[0]["tokens_nostop"][:10]
comparison_table = pd.DataFrame({
    "raw_token": sample_tokens,
    "stemmed": [music_stem(t) for t in sample_tokens],
    "lemmatized": [music_lemmatize(t) for t in sample_tokens],
})
comparison_table

## 7. Assembling the Final Preprocessed Corpus

In [ ]:
# ---------------------------------------------------------------------------
# 7.1 Final document corpora (list-of-tokens, ready for vectorization)
# ---------------------------------------------------------------------------
corpus_stemmed = tokens_df["tokens_stemmed"].tolist()
corpus_lemmatized = tokens_df["tokens_lemmatized"].tolist()
labels = tokens_df["composer"].tolist()

label_encoder = LabelEncoder().fit(COMPOSERS)
y = label_encoder.transform(labels)

print(f"Final corpus: {len(corpus_stemmed)} documents across {len(set(labels))} composers.")
print(f"Example (stemmed) document, first 15 tokens: {corpus_stemmed[0][:15]}")

# Persist for teammates / later sections
with open("preprocessed_corpus.pkl", "wb") as f:
    pickle.dump({
        "corpus_stemmed": corpus_stemmed,
        "corpus_lemmatized": corpus_lemmatized,
        "labels": labels,
        "filepaths": tokens_df["filepath"].tolist(),
    }, f)
print("Saved preprocessed_corpus.pkl")

## 8. Vectorization Method 1 — Bag-of-Words (`CountVectorizer`)

In [ ]:
# ---------------------------------------------------------------------------
# 8.1 Fit CountVectorizer on the lemmatized corpus (chosen as primary; the
#     stemmed corpus is fit identically for the ablation in Section 12-13)
# ---------------------------------------------------------------------------
t0 = time.time()
bow_vectorizer = CountVectorizer(analyzer=lambda tokens: tokens, min_df=2)
X_bow = bow_vectorizer.fit_transform(corpus_lemmatized)
bow_fit_time = time.time() - t0

print(f"Bag-of-Words matrix shape: {X_bow.shape}")
print(f"Vocabulary size (min_df=2): {len(bow_vectorizer.vocabulary_)}")
print(f"Sparsity: {(1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1])):.4%}")
print(f"Fit time: {bow_fit_time:.2f}s")

In [ ]:
# ---------------------------------------------------------------------------
# 8.2 Most frequent tokens overall, and per composer
# ---------------------------------------------------------------------------
vocab_terms = np.array(bow_vectorizer.get_feature_names_out())
total_counts = np.asarray(X_bow.sum(axis=0)).ravel()
top_overall = vocab_terms[np.argsort(total_counts)[::-1][:15]]
print("Top 15 most frequent tokens overall:", list(top_overall))

for composer in COMPOSERS:
    mask = np.array(labels) == composer
    composer_counts = np.asarray(X_bow[mask].sum(axis=0)).ravel()
    top_composer = vocab_terms[np.argsort(composer_counts)[::-1][:10]]
    print(f"\nTop 10 tokens for {composer}: {list(top_composer)}")

## 9. Vectorization Method 2 — TF-IDF (`TfidfVectorizer`)

In [ ]:
# ---------------------------------------------------------------------------
# 9.1 Fit TfidfVectorizer (same tokenization contract as CountVectorizer)
# ---------------------------------------------------------------------------
t0 = time.time()
tfidf_vectorizer = TfidfVectorizer(analyzer=lambda tokens: tokens, min_df=2)
X_tfidf = tfidf_vectorizer.fit_transform(corpus_lemmatized)
tfidf_fit_time = time.time() - t0

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"Sparsity: {(1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1])):.4%}")
print(f"Fit time: {tfidf_fit_time:.2f}s")

In [ ]:
# ---------------------------------------------------------------------------
# 9.2 Most distinctive (highest mean TF-IDF) tokens per composer
# ---------------------------------------------------------------------------
tfidf_terms = np.array(tfidf_vectorizer.get_feature_names_out())

for composer in COMPOSERS:
    mask = np.array(labels) == composer
    mean_tfidf = np.asarray(X_tfidf[mask].mean(axis=0)).ravel()
    top_terms = tfidf_terms[np.argsort(mean_tfidf)[::-1][:10]]
    print(f"Most distinctive tokens for {composer} (by mean TF-IDF): {list(top_terms)}")

## 10. Vectorization Method 3 — Word2Vec (CBOW and Skip-gram)

In [ ]:
# ---------------------------------------------------------------------------
# 10.1 Train CBOW (sg=0) and Skip-gram (sg=1)
# ---------------------------------------------------------------------------
W2V_PARAMS = dict(vector_size=100, window=5, min_count=2, workers=4, epochs=20, seed=SEED)

t0 = time.time()
w2v_cbow = Word2Vec(sentences=corpus_lemmatized, sg=0, **W2V_PARAMS)
cbow_train_time = time.time() - t0

t0 = time.time()
w2v_skipgram = Word2Vec(sentences=corpus_lemmatized, sg=1, **W2V_PARAMS)
skipgram_train_time = time.time() - t0

print(f"CBOW vocabulary size:     {len(w2v_cbow.wv.key_to_index)}   (train time: {cbow_train_time:.2f}s)")
print(f"Skip-gram vocabulary size: {len(w2v_skipgram.wv.key_to_index)}   (train time: {skipgram_train_time:.2f}s)")

In [ ]:
# ---------------------------------------------------------------------------
# 10.2 Intrinsic evaluation: nearest neighbors for a frequent, musically
#      interpretable token (e.g. a common major-triad prime form "0.4.7")
# ---------------------------------------------------------------------------
probe_token = vocab_terms[np.argsort(total_counts)[::-1][0]]  # most frequent lemmatized token
print(f"Probe token (most frequent overall): {probe_token}\n")

if probe_token in w2v_cbow.wv:
    print("CBOW nearest neighbors:")
    for tok, sim in w2v_cbow.wv.most_similar(probe_token, topn=8):
        print(f"  {tok:12s}  cosine similarity = {sim:.3f}")

if probe_token in w2v_skipgram.wv:
    print("\nSkip-gram nearest neighbors:")
    for tok, sim in w2v_skipgram.wv.most_similar(probe_token, topn=8):
        print(f"  {tok:12s}  cosine similarity = {sim:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# 10.3 Document-level Word2Vec features (mean-pooled token embeddings)
# ---------------------------------------------------------------------------
def document_vector(tokens, w2v_model):
    """Average the embeddings of all in-vocabulary tokens in a document.
    Documents with no in-vocabulary tokens fall back to a zero vector."""
    vectors = [w2v_model.wv[t] for t in tokens if t in w2v_model.wv]
    if not vectors:
        return np.zeros(w2v_model.vector_size)
    return np.mean(vectors, axis=0)

X_w2v_cbow = np.vstack([document_vector(doc, w2v_cbow) for doc in corpus_lemmatized])
X_w2v_skipgram = np.vstack([document_vector(doc, w2v_skipgram) for doc in corpus_lemmatized])

print("Document-level CBOW feature matrix:     ", X_w2v_cbow.shape)
print("Document-level Skip-gram feature matrix:", X_w2v_skipgram.shape)

## 11. Embedding Visualization

In [ ]:
# ---------------------------------------------------------------------------
# 11.1 PCA and t-SNE projections of the CBOW token embedding space
# ---------------------------------------------------------------------------
top_n_tokens = 300  # visualize the most frequent tokens only, for legibility
frequent_tokens = vocab_terms[np.argsort(total_counts)[::-1][:top_n_tokens]]
frequent_tokens = [t for t in frequent_tokens if t in w2v_cbow.wv]

embedding_matrix = np.vstack([w2v_cbow.wv[t] for t in frequent_tokens])

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(embedding_matrix)
tsne_2d = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(embedding_matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(pca_2d[:, 0], pca_2d[:, 1], s=10, alpha=0.6)
axes[0].set_title("PCA — CBOW Token Embeddings")

axes[1].scatter(tsne_2d[:, 0], tsne_2d[:, 1], s=10, alpha=0.6)
axes[1].set_title("t-SNE — CBOW Token Embeddings")

plt.tight_layout()
plt.show()

## 12. Comparative Analysis of Vectorization Methods

In [ ]:
# ---------------------------------------------------------------------------
# 12.1 Structural comparison table
# ---------------------------------------------------------------------------
def sparsity(matrix):
    if hasattr(matrix, "nnz"):
        return 1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])
    return float(np.mean(matrix == 0))

structural_comparison = pd.DataFrame([
    {"method": "Bag-of-Words", "dimensionality": X_bow.shape[1],
     "sparsity": sparsity(X_bow), "fit_time_s": bow_fit_time, "representation": "sparse, count-based"},
    {"method": "TF-IDF", "dimensionality": X_tfidf.shape[1],
     "sparsity": sparsity(X_tfidf), "fit_time_s": tfidf_fit_time, "representation": "sparse, weighted-count"},
    {"method": "Word2Vec (CBOW)", "dimensionality": X_w2v_cbow.shape[1],
     "sparsity": sparsity(X_w2v_cbow), "fit_time_s": cbow_train_time, "representation": "dense, learned embedding"},
    {"method": "Word2Vec (Skip-gram)", "dimensionality": X_w2v_skipgram.shape[1],
     "sparsity": sparsity(X_w2v_skipgram), "fit_time_s": skipgram_train_time, "representation": "dense, learned embedding"},
]).set_index("method")

structural_comparison

## 13. Extrinsic Evaluation (Downstream Classification Probe)

In [ ]:
# ---------------------------------------------------------------------------
# 13.1 Cross-validated probe classifier for each representation
# ---------------------------------------------------------------------------
def cv_probe(X, y, name, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    clf = LogisticRegression(max_iter=2000, multi_class="multinomial")
    scores = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")
    print(f"{name:22s}  mean accuracy = {scores.mean():.4f}  (+/- {scores.std():.4f})")
    return scores.mean(), scores.std()

results = {}
results["Bag-of-Words"] = cv_probe(X_bow, y, "Bag-of-Words")
results["TF-IDF"] = cv_probe(X_tfidf, y, "TF-IDF")
results["Word2Vec (CBOW)"] = cv_probe(X_w2v_cbow, y, "Word2Vec (CBOW)")
results["Word2Vec (Skip-gram)"] = cv_probe(X_w2v_skipgram, y, "Word2Vec (Skip-gram)")

In [ ]:
# ---------------------------------------------------------------------------
# 13.2 Stemmed- vs. lemmatized-corpus ablation (using Bag-of-Words as the probe)
# ---------------------------------------------------------------------------
bow_stemmed_vectorizer = CountVectorizer(analyzer=lambda tokens: tokens, min_df=2)
X_bow_stemmed = bow_stemmed_vectorizer.fit_transform(corpus_stemmed)

print("Ablation: stemmed vs. lemmatized corpus (Bag-of-Words features)")
cv_probe(X_bow_stemmed, y, "BoW (stemmed corpus)")
cv_probe(X_bow, y, "BoW (lemmatized corpus)")

In [ ]:
# ---------------------------------------------------------------------------
# 13.3 Visual summary of the extrinsic evaluation
# ---------------------------------------------------------------------------
summary_df = pd.DataFrame(
    [(k, v[0], v[1]) for k, v in results.items()],
    columns=["method", "mean_accuracy", "std_accuracy"]
).sort_values("mean_accuracy", ascending=False)

plt.figure(figsize=(7, 4))
sns.barplot(data=summary_df, x="method", y="mean_accuracy", palette="viridis")
plt.errorbar(x=range(len(summary_df)), y=summary_df["mean_accuracy"],
             yerr=summary_df["std_accuracy"], fmt="none", c="black", capsize=4)
plt.ylabel("5-fold CV Accuracy (Logistic Regression probe)")
plt.title("Extrinsic Comparison of Vectorization Methods")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

summary_df

## 14. Handoff Artifacts for the Team

In [ ]:
import os
import numpy as np
import scipy.sparse as sp
import dill  # Instantly resolves lambda pickling issues

# ---------------------------------------------------------------------------
# 14.1 Persist all pipeline artifacts safely
# ---------------------------------------------------------------------------

# 1. Save scikit-learn vectorizers using dill (handles lambda functions)
with open("bow_vectorizer.pkl", "wb") as f:
    dill.dump(bow_vectorizer, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    dill.dump(tfidf_vectorizer, f)

# 2. Save Gensim Word2Vec models
w2v_cbow.save("w2v_cbow.model")
w2v_skipgram.save("w2v_skipgram.model")

# 3. Save sparse matrices efficiently without blowing up RAM
sp.save_npz("X_bow.npz", X_bow)
sp.save_npz("X_tfidf.npz", X_tfidf)

# 4. Save dense embeddings, targets, and metadata
np.savez(
    "vectorized_dense_features.npz",
    X_w2v_cbow=X_w2v_cbow,
    X_w2v_skipgram=X_w2v_skipgram,
    y=y,
    composer_classes=label_encoder.classes_,
)

print("All handoff artifacts saved successfully!")
print("Saved files:", os.listdir("."))

## 15. References

Blanderbuss. (n.d.). *Classical music MIDI* [Data set]. Kaggle.
https://www.kaggle.com/datasets/blanderbuss/midi-classic-music

Chuan, C.-H., Agres, K., & Herremans, D. (2020). From context to concept:
Exploring semantic relationships in music with word2vec. *Neural Computing
and Applications, 32*, 1023–1036.

Cuthbert, M. S., & Ariza, C. (2010). music21: A toolkit for computer-aided
musicology and symbolic music data. In *Proceedings of the 11th
International Society for Music Information Retrieval Conference*
(pp. 637–642).

Forte, A. (1973). *The structure of atonal music*. Yale University Press.

Herremans, D., Chuan, C.-H., & Chew, E. (2017). A functional taxonomy of
music generation systems. *ACM Computing Surveys, 50*(5), 1–30.

Huang, C.-Z. A., Vaswani, A., Uszkoreit, J., Shazeer, N., Simon, I.,
Hawthorne, C., Dai, A. M., Hoffman, M. D., Dinculescu, M., & Eck, D.
(2019). Music Transformer: Generating music with long-term structure. In
*Proceedings of the International Conference on Learning
Representations*.

Manning, C. D., Raghavan, P., & Schütze, H. (2008). *Introduction to
information retrieval*. Cambridge University Press.

Mikolov, T., Chen, K., Corrado, G., & Dean, J. (2013a). Efficient
estimation of word representations in vector space. *arXiv preprint
arXiv:1301.3781*.

Mikolov, T., Sutskever, I., Chen, K., Corrado, G., & Dean, J. (2013b).
Distributed representations of words and phrases and their
compositionality. *Advances in Neural Information Processing Systems, 26*,
3111–3119.

Miller, G. A. (1995). WordNet: A lexical database for English.
*Communications of the ACM, 38*(11), 39–41.

Oore, S., Simon, I., Dieleman, S., Eck, D., & Simoncelli, E. (2020). This
time with feeling: Learning expressive musical performance. *Neural
Computing and Applications, 32*, 955–967.

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B.,
Grisel, O., et al. (2011). Scikit-learn: Machine learning in Python.
*Journal of Machine Learning Research, 12*, 2825–2830.

Porter, M. F. (1980). An algorithm for suffix stripping. *Program, 14*(3),
130–137.

Raffel, C., & Ellis, D. P. W. (2014). Intuitive analysis, creation and
manipulation of MIDI data with pretty_midi. In *Proceedings of the 15th
International Society for Music Information Retrieval Conference, Late
Breaking and Demo Papers* (pp. 84–93).

Řehůřek, R., & Sojka, P. (2010). Software framework for topic modelling
with large corpora. In *Proceedings of the LREC 2010 Workshop on New
Challenges for NLP Frameworks* (pp. 45–50). [gensim]

Salton, G., & Buckley, C. (1988). Term-weighting approaches in automatic
text retrieval. *Information Processing & Management, 24*(5), 513–523.

van der Maaten, L., & Hinton, G. (2008). Visualizing data using t-SNE.
*Journal of Machine Learning Research, 9*, 2579–2605.
